In [1]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices('GPU'))

2026-05-27 12:09:32.018423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779883772.182313      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779883772.236200      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779883772.612862      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779883772.612903      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779883772.612906      57 computation_placer.cc:177] computation placer alr

TF version: 2.19.0
GPUs visible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
!pip install arch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 26.7 MB/s eta 0:00:0000:01


In [9]:
import os, shutil

DATASET = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data_pipeline.py' in files:
        DATASET = root
        break

if DATASET is None:
    raise RuntimeError(
        "Could not find data_pipeline.py anywhere under /kaggle/input/. "
        "Make sure the .py files are part of your attached dataset.")

print(f"Found dataset at: {DATASET}")
print(f"Files there: {sorted(os.listdir(DATASET))}")

WORK = '/kaggle/working'

# Copy all the .py files into the writable working dir.
for fname in ['data_pipeline.py', 'mmd_noise.py', 'mmd_model.py',
              'diffusion_model.py', 'evaluate.py', 'smoke_test.py']:
    shutil.copy(f'{DATASET}/{fname}', f'{WORK}/{fname}')

# Keep the utils/ folder structure so "from utils.gaussianize import ..." works.
os.makedirs(f'{WORK}/utils', exist_ok=True)
shutil.copy(f'{DATASET}/gaussianize.py', f'{WORK}/utils/gaussianize.py')
open(f'{WORK}/utils/__init__.py', 'a').close()

# And the data folder.
os.makedirs(f'{WORK}/data', exist_ok=True)
shutil.copy(f'{DATASET}/spx_20231229.csv', f'{WORK}/data/spx_20231229.csv')

os.chdir(WORK)
print("\nWorking dir:", os.getcwd())
print("Files:", sorted(os.listdir()))

Found dataset at: /kaggle/input/datasets/annagrenz/masterthesis
Files there: ['data_pipeline.py', 'diffusion_model.py', 'evaluate.py', 'gaussianize.py', 'mmd_model.py', 'mmd_noise.py', 'smoke_test.py', 'spx_20231229.csv']

Working dir: /kaggle/working
Files: ['.virtual_documents', 'data', 'data_pipeline.py', 'diffusion_model.py', 'evaluate.py', 'mmd_model.py', 'mmd_noise.py', 'smoke_test.py', 'utils']


In [10]:
# Reduce MMD batch size to fit in P100 GPU memory
with open('/kaggle/working/mmd_model.py', 'r') as f:
    code = f.read()

old = 'BATCH_SIZE          = 38'
new = 'BATCH_SIZE          = 16'

if old in code:
    code = code.replace(old, new)
    with open('/kaggle/working/mmd_model.py', 'w') as f:
        f.write(code)
    print('✓ Reduced BATCH_SIZE from 38 to 16')
else:
    if new in code:
        print('Already patched.')
    else:
        print('⚠️  Could not find the BATCH_SIZE line. Maybe spacing is different?')

# Clear bytecode cache so the new value gets picked up
import shutil, os
if os.path.exists('/kaggle/working/__pycache__'):
    shutil.rmtree('/kaggle/working/__pycache__')
    print('✓ Cleared __pycache__')

✓ Reduced BATCH_SIZE from 38 to 16


In [11]:
!python smoke_test.py

Smoke tests — verify everything is wired correctly

[TEST] Data pipeline loads SPX
  → SPX rows: 7299, train returns: 5970, oos returns: 1327
  → MMD windows: (114, 300, 2),  diff windows: (286, 256)
  ✓ PASSED

[TEST] Lambert W Gaussianisation works
  → input  kurtosis (excess) ≈ 8.53
  → output kurtosis (excess) ≈ -0.00
  → fitted μ=+0.10004, σ=3.71927
  ✓ PASSED

[TEST] MA(20) noise sampler produces noise
  → noise shape (8, 299, 4), mean=+0.0000, std=0.5939
  ✓ PASSED

[TEST] MMD generator forward pass
2026-05-27 12:18:13.775125: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779884293.797654     207 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779884293.805665     207 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting 

In [12]:
!python diffusion_model.py

2026-05-27 12:19:10.935271: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779884350.959050     312 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779884350.967053     312 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779884350.986651     312 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779884350.986679     312 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779884350.986682     312 computation_placer.cc:177] computation placer alr

In [19]:
!python mmd_model.py 
# A second training run verified reproducibility of the
# original MMD-Signature results. The model showed the same convergence
# pattern: rapid initial descent (MMD ~4000 → ~10 over the first 50 epochs),
# followed by a slowly-improving plateau. The Kaggle notebook display stopped flushing
# output for an extended period and the run was eventually interrupted
# manually at epoch ~1530, but the training itself proceeded normally.
#
# The thesis evaluation uses the trained weights from the FIRST training
# run (best_model.weights.h5, dated 2026-05-14), which exhibited the same
# convergence behaviour. Both runs are consistent.

2026-05-27 12:42:35.655866: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779885755.678899    1160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779885755.686441    1160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779885755.705750    1160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779885755.705805    1160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779885755.705813    1160 computation_placer.cc:177] computation placer alr

In [4]:
import os
print(os.listdir('/kaggle/input'))

['datasets']


In [13]:
import shutil, os

# Copy ONLY the MMD weights — leave your new diffusion_paper weights untouched.
shutil.copytree('/kaggle/input/datasets/annagrenz/weights/runs/mmd_paper',
                '/kaggle/working/runs/mmd_paper',
                dirs_exist_ok=True)

# Verify
print(os.listdir('/kaggle/working/runs/mmd_paper'))
# Should print: ['best_model.weights.h5', 'results.png']

['best_model.weights.h5', 'results.png']


In [14]:
!python evaluate.py

2026-05-27 12:28:16.482812: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779884896.506128    1048 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779884896.514376    1048 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779884896.533835    1048 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779884896.533895    1048 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779884896.533902    1048 computation_placer.cc:177] computation placer alr

In [15]:
import os
print('Files in runs/:')
for root, _, files in os.walk('runs'):
    for f in files:
        path = os.path.join(root, f)
        print(f'  {path}  ({os.path.getsize(path):,} bytes)')

Files in runs/:
  runs/diffusion_paper/weights_ep0060.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0070.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0050.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_final.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0030.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0020.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0040.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/results.png  (100,758 bytes)
  runs/diffusion_paper/weights_ep0080.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0010.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0100.weights.h5  (89,178,912 bytes)
  runs/diffusion_paper/weights_ep0090.weights.h5  (89,178,912 bytes)
  runs/mmd_paper/best_model.weights.h5  (86,680 bytes)
  runs/mmd_paper/results.png  (190,386 bytes)
  runs/evaluation/oos/stylised_facts.png  (480,649 bytes)
  runs/evaluati

In [17]:
import os

print('Contents of runs/:')
for entry in sorted(os.listdir('runs')):
    full = os.path.join('runs', entry)
    if os.path.isdir(full):
        print(f'  📁 {entry}/')
        for sub in sorted(os.listdir(full)):
            sub_full = os.path.join(full, sub)
            size = os.path.getsize(sub_full)
            print(f'      {sub}  ({size:,} bytes)')
    else:
        print(f'  📄 {entry}')

print('\nContents of runs/evaluation/ (if it exists):')
if os.path.exists('runs/evaluation'):
    for root, dirs, files in os.walk('runs/evaluation'):
        for f in files:
            path = os.path.join(root, f)
            print(f'  {path}  ({os.path.getsize(path):,} bytes)')
else:
    print('  (no evaluation folder found)')

Contents of runs/:
  📁 diffusion_paper/
      results.png  (100,758 bytes)
      weights_ep0010.weights.h5  (89,178,912 bytes)
      weights_ep0020.weights.h5  (89,178,912 bytes)
      weights_ep0030.weights.h5  (89,178,912 bytes)
      weights_ep0040.weights.h5  (89,178,912 bytes)
      weights_ep0050.weights.h5  (89,178,912 bytes)
      weights_ep0060.weights.h5  (89,178,912 bytes)
      weights_ep0070.weights.h5  (89,178,912 bytes)
      weights_ep0080.weights.h5  (89,178,912 bytes)
      weights_ep0090.weights.h5  (89,178,912 bytes)
      weights_ep0100.weights.h5  (89,178,912 bytes)
      weights_final.weights.h5  (89,178,912 bytes)
  📁 evaluation/
      oos  (4,096 bytes)
      train  (4,096 bytes)
  📁 mmd_paper/
      best_model.weights.h5  (86,680 bytes)
      results.png  (190,386 bytes)

Contents of runs/evaluation/ (if it exists):
  runs/evaluation/oos/stylised_facts.png  (480,649 bytes)
  runs/evaluation/oos/mmd_comparison.png  (36,317 bytes)
  runs/evaluation/oos/gain_loss

In [18]:
import os, zipfile

# Files that actually matter for the thesis
essential = [
    'runs/mmd_paper/best_model.weights.h5',
    'runs/mmd_paper/results.png',
    'runs/diffusion_paper/weights_final.weights.h5',
    'runs/diffusion_paper/results.png',
    'runs/evaluation/train/stylised_facts.png',
    'runs/evaluation/train/gain_loss_endpoint.png',
    'runs/evaluation/train/mmd_comparison.png',
    'runs/evaluation/oos/stylised_facts.png',
    'runs/evaluation/oos/gain_loss_endpoint.png',
    'runs/evaluation/oos/mmd_comparison.png',
]

with zipfile.ZipFile('thesis_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in essential:
        if os.path.exists(f):
            zf.write(f)
            print(f'  added: {f}  ({os.path.getsize(f):,} bytes)')
        else:
            print(f'  MISSING: {f}')

size = os.path.getsize('thesis_outputs.zip')
print(f'\n✓ Created thesis_outputs.zip  ({size:,} bytes, ~{size/1024/1024:.1f} MB)')

  added: runs/mmd_paper/best_model.weights.h5  (86,680 bytes)
  added: runs/mmd_paper/results.png  (190,386 bytes)
  added: runs/diffusion_paper/weights_final.weights.h5  (89,178,912 bytes)
  added: runs/diffusion_paper/results.png  (100,758 bytes)
  added: runs/evaluation/train/stylised_facts.png  (459,360 bytes)
  added: runs/evaluation/train/gain_loss_endpoint.png  (102,869 bytes)
  added: runs/evaluation/train/mmd_comparison.png  (39,405 bytes)
  added: runs/evaluation/oos/stylised_facts.png  (480,649 bytes)
  added: runs/evaluation/oos/gain_loss_endpoint.png  (95,883 bytes)
  added: runs/evaluation/oos/mmd_comparison.png  (36,317 bytes)

✓ Created thesis_outputs.zip  (82,984,696 bytes, ~79.1 MB)
